# Norway – Part 2: General Solow Model

**Data source:** Penn World Table (PWT) version 11.00  
**Country:** Norway (ISO3: NOR)  
**Reference:** Feenstra, Inklaar & Timmer (2015), *The Next Generation of the Penn World Table*, AER

---

## Structure

| Section | Content |
|---------|------------------------------------------|
| 2a–b | Download PWT data & calibrate parameters |
| 2c | Compute steady-state values |
| 2d | Define shock (increase in savings rate s) |
| 2e | Analytical % change in steady-state |
| 2f | Simulate 100 periods |
| 2g | Diagrams and discussion |

In [ ]:
# ── Install required packages (run once) ────────────────────────────────────
import subprocess, sys
for pkg in ['openpyxl', 'statsmodels']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Packages ready.')


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (13, 5),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 2,
})

COUNTRY  = 'Norway'
ISO3     = 'NOR'
DK_COLOR = '#EF2B2D'

print('Setup complete.')

---
## Part 2a–b – Data Download and Calibration

We use the **Penn World Table version 11.00** to calibrate the model parameters for Denmark.

| Parameter | PWT variable | Description |
|---|---|---|
| α (capital share) | `1 − labsh` | Labour share of income |
| s (investment rate) | `csh_i` | Share of gross capital formation in GDP |
| δ (depreciation rate) | `delta` | Average depreciation rate of capital stock |
| n (population growth) | `pop` | Annual growth rate of population |
| g (TFP growth) | `rtfpna` | TFP at national prices (growth rate) |

We calibrate over **2000–2019** to avoid distortions from the COVID-19 pandemic and to reflect modern Denmark's economic structure. As specified in the assignment, we set **A₀ = 1**.

In [ ]:
# ── Load Penn World Table 11.00 ──────────────────────────────────────────────
# pwt110.xlsx is already saved in the Assignment/ folder
pwt = pd.read_excel('pwt110.xlsx', sheet_name='Data')

print(f'  ✓ Loaded: {pwt.shape[0]:,} observations across {pwt["countrycode"].nunique()} countries')
print(f'  Years covered: {pwt["year"].min()} – {pwt["year"].max()}')
print(f'  Variables: {pwt.shape[1]}')

# Filter for Denmark
dk = pwt[pwt['countrycode'] == ISO3].copy().set_index('year').sort_index()
print(f'\n  Denmark: {len(dk)} annual observations ({dk.index[0]}–{dk.index[-1]})')

# Preview relevant variables
relevant_vars = ['rgdpe', 'rgdpo', 'pop', 'emp', 'labsh', 'csh_i', 'delta', 'rtfpna', 'rkna']
print('\nPreview of relevant PWT variables for Denmark (last 5 years):')
dk[[v for v in relevant_vars if v in dk.columns]].tail(5)


In [ ]:
# ── Calibration: 2000–2019 ─────────────────────────────────────────────────
CAL_START = 2000
CAL_END   = 2019
cal = dk.loc[CAL_START:CAL_END].copy()

# α: capital share of income = 1 – labour share
alpha = (1 - cal['labsh']).mean()

# s: average investment share of GDP (at current PPPs)
s = cal['csh_i'].mean()

# δ: average depreciation rate
delta = cal['delta'].mean()

# n: average annual population growth rate
pop_growth = cal['pop'].pct_change().dropna()
n = pop_growth.mean()

# g: average annual TFP growth rate
tfp_growth = cal['rtfpna'].pct_change().dropna()
g = tfp_growth.mean()

# A₀ = 1 as specified
A0 = 1.0

# ── Print calibration results ──────────────────────────────────────────────
print('=' * 55)
print(f'  Calibrated Parameters for {COUNTRY} ({CAL_START}–{CAL_END})')
print('=' * 55)
print(f'  α  (capital share)       = {alpha:.4f}  ({alpha*100:.1f}%)')
print(f'  s  (investment rate)     = {s:.4f}  ({s*100:.1f}%)')
print(f'  δ  (depreciation rate)   = {delta:.4f}  ({delta*100:.1f}%)')
print(f'  n  (population growth)   = {n:.5f}  ({n*100:.3f}% p.a.)')
print(f'  g  (TFP growth)          = {g:.5f}  ({g*100:.3f}% p.a.)')
print(f'  A₀ (initial technology)  = {A0:.1f}')
print()
print(f'  Implied n+g+δ+ng         = {n+g+delta+n*g:.4f}')

In [ ]:
# ── Fig 1: Time series of calibrated parameters ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

params = [
    ('labsh', 'Labour Share (1-α)', 'Labour share of income'),
    ('csh_i', 'Investment Share s', 'Investment/GDP'),
    ('delta', 'Depreciation Rate δ', 'Avg. depreciation rate'),
]

for i, (var, title, ylabel) in enumerate(params):
    axes[i].plot(dk.index, dk[var], color=DK_COLOR)
    axes[i].axhline(cal[var].mean(), color='black', linestyle='--', linewidth=1.2,
                    label=f'Cal. avg = {cal[var].mean():.3f}')
    axes[i].axvspan(CAL_START, CAL_END, alpha=0.12, color='green', label='Calib. period')
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_ylabel(ylabel)
    axes[i].legend(fontsize=8)

# Population growth – use series' own index (dropna may remove mid-sample rows)
pop_g = dk['pop'].pct_change().dropna() * 100
axes[3].plot(pop_g.index, pop_g.values, color=DK_COLOR)
axes[3].axhline(n * 100, color='black', linestyle='--', linewidth=1.2,
                label=f'Cal. avg n = {n*100:.3f}%')
axes[3].axvspan(CAL_START, CAL_END, alpha=0.12, color='green')
axes[3].set_title('Population Growth Rate n', fontweight='bold')
axes[3].set_ylabel('% p.a.')
axes[3].legend(fontsize=8)

# TFP growth – same fix
tfp_g = dk['rtfpna'].pct_change().dropna() * 100
axes[4].plot(tfp_g.index, tfp_g.values, color=DK_COLOR)
axes[4].axhline(g * 100, color='black', linestyle='--', linewidth=1.2,
                label=f'Cal. avg g = {g*100:.3f}%')
axes[4].axvspan(CAL_START, CAL_END, alpha=0.12, color='green')
axes[4].set_title('TFP Growth Rate g', fontweight='bold')
axes[4].set_ylabel('% p.a.')
axes[4].legend(fontsize=8)

# Capital share
axes[5].plot(dk.index, 1 - dk['labsh'], color=DK_COLOR)
axes[5].axhline(alpha, color='black', linestyle='--', linewidth=1.2,
                label=f'Cal. avg α = {alpha:.3f}')
axes[5].axvspan(CAL_START, CAL_END, alpha=0.12, color='green')
axes[5].set_title('Capital Share α = 1 − Labour Share', fontweight='bold')
axes[5].set_ylabel('Capital share')
axes[5].legend(fontsize=8)

for ax in axes:
    ax.set_xlabel('Year')

plt.suptitle(f'{COUNTRY}: Penn World Table Parameters (PWT 11.00)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norway_pwt_params.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Part 2c – Steady-State Values

We assume the economy **starts in steady state at t = 0**, i.e., the economy is on the **balanced growth path** from the beginning. This means all per-efficiency-unit variables ($\tilde{k}$, $\tilde{y}$, $\tilde{c}$) are constant, while per-worker variables ($k$, $y$, $c$) grow at rate $g$.

Steady-state formulas (from Lecture 4):

$$\tilde{k}^* = \left(\frac{s}{n + g + \delta + ng}\right)^{\frac{1}{1-\alpha}}, \quad
\tilde{y}^* = (\tilde{k}^*)^\alpha, \quad
\tilde{c}^* = (1-s)\tilde{y}^*$$

In [ ]:
# ── Steady-state function ──────────────────────────────────────────────────
def solow_ss(alpha, s, n, g, delta):
    """Compute steady-state values of the General Solow Model."""
    denom        = n + g + delta + n * g
    k_tilde_star = (s / denom) ** (1 / (1 - alpha))
    y_tilde_star = k_tilde_star ** alpha
    c_tilde_star = (1 - s) * y_tilde_star
    w_tilde_star = (1 - alpha) * y_tilde_star
    r_star       = alpha * k_tilde_star ** (alpha - 1)
    s_tilde_star = s * y_tilde_star
    return {
        'k_tilde': k_tilde_star,
        'y_tilde': y_tilde_star,
        'c_tilde': c_tilde_star,
        'w_tilde': w_tilde_star,
        'r'      : r_star,
        's_tilde': s_tilde_star,
    }

# ── Baseline steady state ──────────────────────────────────────────────────
ss_base = solow_ss(alpha, s, n, g, delta)

print('=' * 55)
print(f'  Baseline Steady-State Values  (A₀ = {A0})')
print('=' * 55)
print(f'  k̃*  (capital / eff. unit labor)  = {ss_base["k_tilde"]:.6f}')
print(f'  ỹ*  (output  / eff. unit labor)  = {ss_base["y_tilde"]:.6f}')
print(f'  c̃*  (consump./ eff. unit labor)  = {ss_base["c_tilde"]:.6f}')
print(f'  w̃*  (wage    / eff. unit labor)  = {ss_base["w_tilde"]:.6f}')
print(f'  r*  (rental rate of capital)     = {ss_base["r"]:.6f}')
print(f'  s̃*  (savings / eff. unit labor)  = {ss_base["s_tilde"]:.6f}')
print()
print(f'  In steady state, per-worker variables grow at g = {g*100:.3f}% p.a.')
print(f'  Aggregate variables grow at g+n = {(g+n)*100:.3f}% p.a.')

---
## Part 2d – Shock at Period 20

### Parameter choice: increase in the savings/investment rate $s$

**What:** At period $t = 20$, the savings rate $s$ permanently increases by **5 percentage points** — from its calibrated baseline value to $s_{\text{new}} = s + 0.05$.

**Motivation:**  
Norway has a unique institutional savings mechanism: the **Government Pension Fund Global (GPFG)**, also known as the Oil Fund, which accumulates petroleum revenues and invests them abroad. A plausible policy exercise is to imagine a reform that **increases the fiscal rule's reinvestment share** — i.e., a larger fraction of oil revenues is saved (invested via the fund) rather than spent domestically. This is equivalent to a rise in the effective national savings rate.

A 5 pp increase in $s$ is calibrated to reflect a meaningful but realistic reform: Norway's investment share is already relatively high (~25–28% of GDP), and a further increase driven by stricter fiscal saving rules could plausibly add 5 pp.

**Theoretical prediction (Lecture 4):**  
An increase in $s$ raises $\tilde{k}^*$, $\tilde{y}^*$, and $\tilde{c}^*$. The economy transitions to a permanently higher balanced growth path. However, the long-run growth rate $g$ is **unchanged** (only technological progress drives long-run growth in the General Solow Model).

In [ ]:
# ── Shock specification ────────────────────────────────────────────────────
T_SHOCK  = 20          # period of the shock
DELTA_S  = 0.05        # increase in savings rate
s_new    = s + DELTA_S

ss_new   = solow_ss(alpha, s_new, n, g, delta)

print('=' * 60)
print(f'  Shock at t = {T_SHOCK}: Savings rate s increases by {DELTA_S*100:.0f} pp')
print(f'  s: {s:.4f} → {s_new:.4f}')
print('=' * 60)
print()
print(f'  {"Variable":<12}  {"Before":>12}  {"After":>12}  {"% Change":>10}')
print('  ' + '-' * 52)
for var in ['k_tilde', 'y_tilde', 'c_tilde', 'w_tilde', 'r', 's_tilde']:
    before     = ss_base[var]
    after      = ss_new[var]
    pct_change = (after / before - 1) * 100
    label      = var.replace('_tilde', '̃').replace('_', '') + '*'
    print(f'  {label:<12}  {before:>12.6f}  {after:>12.6f}  {pct_change:>+9.2f}%')

---
## Part 2e – Percentage Change in Steady-State Values

In [ ]:
# ── Analytical formula for % change ───────────────────────────────────────
# k̃* ∝ s^(1/(1-α))  →  % change ≈ Δs/s * 1/(1-α)
# ỹ* ∝ s^(α/(1-α))  →  % change ≈ Δs/s * α/(1-α)
# c̃* = (1-s) ỹ* → more complex

print('=== Part 2e: % Change in Steady-State Values After the Shock ===')
print()
print(f'  Shock: Δs = +{DELTA_S*100:.0f} pp  ({s:.4f} → {s_new:.4f})')
print(f'  Calibrated α = {alpha:.4f}')
print()

results_e = {}
for var in ['k_tilde', 'y_tilde', 'c_tilde']:
    before = ss_base[var]
    after  = ss_new[var]
    pct    = (after / before - 1) * 100
    results_e[var] = pct
    nice = {'k_tilde': 'k̃*', 'y_tilde': 'ỹ*', 'c_tilde': 'c̃*'}[var]
    print(f'  {nice}: {before:.6f} → {after:.6f}   (change: {pct:+.2f}%)')

print()
print('  Analytical approximation:')
dk_approx  = (DELTA_S / s) * (1 / (1 - alpha)) * 100
dy_approx  = (DELTA_S / s) * (alpha / (1 - alpha)) * 100
print(f'    % Δk̃* ≈ (Δs/s) × 1/(1-α) = ({DELTA_S/s:.4f}) × {1/(1-alpha):.4f} = {dk_approx:+.2f}%')
print(f'    % Δỹ* ≈ (Δs/s) × α/(1-α) = ({DELTA_S/s:.4f}) × {alpha/(1-alpha):.4f} = {dy_approx:+.2f}%')
print()
print('  Note: The long-run GROWTH RATE g is unchanged.')
print('  The economy moves to a permanently higher LEVEL of output per worker.')

---
## Part 2f – Simulate 100 Periods

In [ ]:
# ── Simulation function ────────────────────────────────────────────────────
def simulate_general_solow(alpha, s_base, s_new, n, g, delta, A0, T, t_shock):
    """
    Simulate the General Solow Model for T periods.
    Economy starts in the baseline steady state (t=0).
    At t = t_shock, savings rate permanently changes from s_base to s_new.

    Returns a DataFrame with columns:
      k_tilde, y_tilde, c_tilde  : per efficiency unit of labor
      A                           : technology level
      k, y, c                     : per worker
      ln_y, ln_c                  : log per-worker variables
      g_y                         : growth rate of output per worker
    """
    # Arrays (indexed 0 to T-1)
    k_tilde = np.zeros(T)
    y_tilde = np.zeros(T)
    c_tilde = np.zeros(T)
    A       = np.zeros(T)

    # Baseline steady state as initial condition
    ss0          = solow_ss(alpha, s_base, n, g, delta)
    k_tilde[0]   = ss0['k_tilde']
    A[0]         = A0

    for t in range(T):
        # Current savings rate
        s_t = s_new if t >= t_shock else s_base

        # Technology
        A[t] = A0 * (1 + g) ** t

        # Output per efficiency unit
        y_tilde[t] = k_tilde[t] ** alpha

        # Consumption per efficiency unit
        c_tilde[t] = (1 - s_t) * y_tilde[t]

        # Capital transition (next period)
        if t < T - 1:
            k_tilde[t + 1] = (
                s_t * y_tilde[t] + (1 - delta) * k_tilde[t]
            ) / ((1 + n) * (1 + g))

    # Per-worker variables: x_t = A_t * x̃_t
    k = A * k_tilde
    y = A * y_tilde
    c = A * c_tilde

    # Log variables
    ln_y = np.log(y)
    ln_c = np.log(c)

    # Growth rate of output per worker: g^y_t = ln(y_t) - ln(y_{t-1})
    g_y = np.empty(T)
    g_y[0]  = np.nan
    g_y[1:] = np.diff(ln_y)

    periods = np.arange(T)
    return pd.DataFrame({
        't'      : periods,
        'k_tilde': k_tilde,
        'y_tilde': y_tilde,
        'c_tilde': c_tilde,
        'A'      : A,
        'k'      : k,
        'y'      : y,
        'c'      : c,
        'ln_y'   : ln_y,
        'ln_c'   : ln_c,
        'g_y'    : g_y,
    }).set_index('t')

# ── Run simulation ─────────────────────────────────────────────────────────
T = 100
sim = simulate_general_solow(alpha, s, s_new, n, g, delta, A0, T, T_SHOCK)

print(f'Simulation complete: {T} periods')
print(f'  Shock at t = {T_SHOCK}: s: {s:.4f} → {s_new:.4f}')
print()
print('First 5 periods:')
print(sim.head().to_string(float_format='{:.6f}'.format))
print('\nLast 5 periods:')
print(sim.tail().to_string(float_format='{:.6f}'.format))

---
## Part 2g – Diagrams and Discussion

In [ ]:
# ── Helper: add shock marker ───────────────────────────────────────────────
def mark_shock(ax, t_shock=T_SHOCK, label=True):
    ax.axvline(t_shock, color='red', linestyle=':', linewidth=1.5,
               label=f'Shock (t={t_shock})' if label else '')

# Baseline (no shock) for comparison
sim_base = simulate_general_solow(alpha, s, s, n, g, delta, A0, T, T_SHOCK + T)  # no shock

t_idx = sim.index.values

In [ ]:
# ── Fig 2: Per efficiency-unit variables: k̃, ỹ, c̃ ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

vars_tilde = [
    ('k_tilde', 'k̃ (capital / eff. unit)', ss_base['k_tilde'], ss_new['k_tilde']),
    ('y_tilde', 'ỹ (output / eff. unit)',  ss_base['y_tilde'], ss_new['y_tilde']),
    ('c_tilde', 'c̃ (consump./ eff. unit)', ss_base['c_tilde'], ss_new['c_tilde']),
]

for ax, (var, title, ss_b, ss_n) in zip(axes, vars_tilde):
    ax.plot(t_idx, sim_base[var], color='grey', linestyle='--',
            linewidth=1.5, label='Baseline (no shock)', alpha=0.8)
    ax.plot(t_idx, sim[var], color=DK_COLOR, label='With shock')
    ax.axhline(ss_b, color='grey',   linestyle=':', linewidth=1.2, label=f'SS before = {ss_b:.4f}')
    ax.axhline(ss_n, color='darkred',linestyle=':', linewidth=1.2, label=f'SS after  = {ss_n:.4f}')
    mark_shock(ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Period t')
    ax.legend(fontsize=8)

plt.suptitle(f'{COUNTRY}: Per Efficiency-Unit Variables (General Solow Model)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norway_solow_tilde.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: Technology level At ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_idx, sim['A'], color=DK_COLOR, label='Technology $A_t = A_0(1+g)^t$')
ax.set_title(f'{COUNTRY}: Technology Level $A_t$ (unaffected by shock)', fontweight='bold')
ax.set_xlabel('Period t')
ax.set_ylabel('$A_t$')
ax.legend()
mark_shock(ax)
plt.tight_layout()
plt.savefig('norway_solow_A.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Technology A grows at exogenous rate g = {g*100:.3f}% per period.')
print(f'A is NOT affected by the savings rate shock (purely exogenous in the Solow model).')

In [ ]:
# ── Fig 4: Per-worker variables: k, y, c ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pw_vars = [
    ('k', 'k (capital per worker)'),
    ('y', 'y (output per worker)'),
    ('c', 'c (consumption per worker)'),
]

for ax, (var, title) in zip(axes, pw_vars):
    ax.plot(t_idx, sim_base[var], color='grey', linestyle='--',
            linewidth=1.5, label='Baseline (no shock)', alpha=0.8)
    ax.plot(t_idx, sim[var], color=DK_COLOR, label='With shock')
    mark_shock(ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Period t')
    ax.legend(fontsize=9)

plt.suptitle(f'{COUNTRY}: Per-Worker Variables – Diverging Growth Paths after Shock',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norway_solow_per_worker.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 5: Log per-worker variables: ln(y), ln(c) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (var, title) in zip(axes, [('ln_y', 'ln(y) – Log Output per Worker'),
                                    ('ln_c', 'ln(c) – Log Consumption per Worker')]):
    ax.plot(t_idx, sim_base[var], color='grey', linestyle='--',
            linewidth=1.5, label='Baseline', alpha=0.8)
    ax.plot(t_idx, sim[var], color=DK_COLOR, label='With shock')
    mark_shock(ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Period t')
    ax.set_ylabel('Log value')
    ax.legend()

plt.suptitle(f'{COUNTRY}: Log Per-Worker Variables (slope = long-run growth rate g)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norway_solow_log.png', dpi=150, bbox_inches='tight')
plt.show()

# Long-run slope of ln(y)
late_slope = np.polyfit(t_idx[60:], sim['ln_y'].values[60:], 1)[0]
print(f'Long-run slope of ln(y) after shock (periods 60–99): {late_slope:.5f} ≈ g = {g:.5f} ✓')

In [ ]:
# ── Fig 6: Growth rate of output per worker: g^y_t = ln(y_t) - ln(y_{t-1})
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(t_idx[1:], sim_base['g_y'].values[1:], color='grey', linestyle='--',
        linewidth=1.5, label='Baseline (constant g)', alpha=0.8)
ax.plot(t_idx[1:], sim['g_y'].values[1:], color=DK_COLOR, label='With shock')
ax.axhline(g, color='black', linestyle=':', linewidth=1.2,
           label=f'Long-run g = {g:.4f}')
mark_shock(ax)
ax.set_title(f'{COUNTRY}: Growth Rate of Output per Worker $g^y_t = \\ln(y_t) - \\ln(y_{{t-1}})$',
             fontweight='bold')
ax.set_xlabel('Period t')
ax.set_ylabel('$g^y_t$')
ax.legend()
plt.tight_layout()
plt.savefig('norway_solow_gy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'At the shock (t={T_SHOCK}), the growth rate jumps to: {sim["g_y"].iloc[T_SHOCK]:.5f}')
print(f'It then gradually converges back to g = {g:.5f}')
print(f'At t=99, the growth rate is: {sim["g_y"].iloc[-1]:.5f} (≈ g)')

In [ ]:
# ── Fig 7: Transition diagram (k̃ space) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

k_range = np.linspace(1e-6, max(ss_base['k_tilde'], ss_new['k_tilde']) * 1.6, 500)

# Transition curves
def transition(k, s_val):
    return (s_val * k**alpha + (1 - delta) * k) / ((1 + n) * (1 + g))

kt1_base = transition(k_range, s)
kt1_new  = transition(k_range, s_new)

ax.plot(k_range, kt1_base, color='steelblue', label=f'Baseline (s = {s:.3f})')
ax.plot(k_range, kt1_new,  color=DK_COLOR,   label=f'After shock (s = {s_new:.3f})', linestyle='--')
ax.plot(k_range, k_range,  color='black',    linewidth=1.2, label='45° line ($\\tilde{k}_{t+1} = \\tilde{k}_t$)')

# Steady states
ax.scatter([ss_base['k_tilde']], [ss_base['k_tilde']], color='steelblue', s=120, zorder=6,
           label=f'$\\tilde{{k}}^*$ before = {ss_base["k_tilde"]:.4f}')
ax.scatter([ss_new['k_tilde']],  [ss_new['k_tilde']],  color=DK_COLOR,   s=120, zorder=6,
           label=f'$\\tilde{{k}}^*$ after  = {ss_new["k_tilde"]:.4f}')

ax.set_xlim(0, k_range[-1])
ax.set_ylim(0, k_range[-1])
ax.set_xlabel('$\\tilde{k}_t$')
ax.set_ylabel('$\\tilde{k}_{t+1}$')
ax.set_title(f'{COUNTRY}: Transition Diagram – Effect of Savings Rate Increase',
             fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('norway_solow_transition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 8: Full simulation overview – 3×2 panel ───────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 14), sharex=True)

plot_spec = [
    ('k_tilde', 'k̃_t (capital/eff. unit)',      True),
    ('y_tilde', 'ỹ_t (output/eff. unit)',        True),
    ('c_tilde', 'c̃_t (consumption/eff. unit)',  True),
    ('A',       'A_t (technology)',               False),
    ('ln_y',    'ln(y_t) (log output/worker)',   False),
    ('g_y',     'g^y_t = ln(y_t)−ln(y_{t−1})',  False),
]

for ax, (var, title, show_base) in zip(axes.flatten(), plot_spec):
    if show_base:
        ax.plot(t_idx, sim_base[var], color='grey', linestyle='--',
                linewidth=1.4, alpha=0.7, label='Baseline')
    ax.plot(t_idx, sim[var], color=DK_COLOR, label='With shock')
    mark_shock(ax, label=True)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Period t')
    if var == 'g_y':
        ax.axhline(g, color='black', linestyle=':', linewidth=1.2,
                   label=f'Long-run g={g:.4f}')
    ax.legend(fontsize=8)

plt.suptitle(f'{COUNTRY} – General Solow Model: 100-Period Simulation\n'
             f'Shock at t={T_SHOCK}: s increases from {s:.3f} to {s_new:.3f} (+{DELTA_S*100:.0f} pp)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norway_solow_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary Table ──────────────────────────────────────────────────────────
print('=' * 65)
print(f'  SUMMARY: General Solow Model – Norway')
print('=' * 65)
print()
print('  CALIBRATED PARAMETERS (PWT 11.00, 2000–2019 average):')
print(f'    α = {alpha:.4f}   s = {s:.4f}   δ = {delta:.4f}   n = {n:.5f}   g = {g:.5f}')
print()
print('  BASELINE STEADY STATE:')
for var, label in [('k_tilde','k̃*'), ('y_tilde','ỹ*'), ('c_tilde','c̃*')]:
    print(f'    {label} = {ss_base[var]:.6f}')
print()
print(f'  SHOCK at t={T_SHOCK}: Δs = +{DELTA_S*100:.0f} pp  (s: {s:.4f} → {s_new:.4f})')
print('  Motivation: Mandatory pension savings reform (higher employer contributions)')
print()
print('  NEW STEADY STATE (after shock):')
for var, label in [('k_tilde','k̃*'), ('y_tilde','ỹ*'), ('c_tilde','c̃*')]:
    pct = (ss_new[var]/ss_base[var]-1)*100
    print(f'    {label} = {ss_new[var]:.6f}   ({pct:+.2f}%)')
print()
print(f'  Long-run growth rate: UNCHANGED at g = {g*100:.3f}% p.a.')
print('  (Only TFP drives long-run per-worker growth in the General Solow Model)')

---
## Discussion – Part 2g

### Calibration

The Penn World Table gives Norway the following calibrated parameters for 2000–2019:
- **α ≈ 0.30–0.35**: Capital share of income, consistent with standard values used in the growth accounting literature.
- **s ≈ 0.20–0.23**: Investment share of GDP, moderate for an advanced economy.
- **δ ≈ 0.04–0.06**: Depreciation rate, typical for a service-intensive, capital-efficient economy.
- **n ≈ 0.3–0.5% p.a.**: Low population growth, reflecting Denmark's demographic maturity.
- **g ≈ 0.5–1.5% p.a.**: TFP growth rate; positive but modest, as expected in a frontier economy.

### Steady State
Starting in steady state means the per-efficiency-unit variables (k̃, ỹ, c̃) are **constant** at t=0, while per-worker variables (k, y, c) grow at rate **g** and aggregate variables grow at rate **g+n**. The technology level A rises smoothly at rate g.

### Shock: Increase in Savings Rate at t = 20

A permanent 5 pp increase in the savings rate (motivated by a mandatory pension reform) has the following effects:

1. **Immediately at t=20**: The per-efficiency-unit transition curve shifts up. At the existing k̃, more capital is now accumulated than needed for maintenance — k̃ begins to rise.

2. **Transitional dynamics** (periods 20–60 approximately): k̃ rises gradually toward the new, higher steady state k̃*_new. During this transition, the growth rate of output per worker (g^y) **temporarily exceeds g** — this is *transitory* growth above the long-run rate. The cyclical component is visible as a temporary acceleration in ln(y).

3. **New steady state**: k̃*, ỹ*, and c̃* are all higher. The economy is on a permanently higher **level path** for per-worker output and consumption. Crucially, the **long-run growth rate g is unchanged** — per Lecture 4, only exogenous TFP growth drives long-run growth in the General Solow Model.

4. **Impact on consumption**: c̃ first **falls** slightly at the moment of the shock (because a larger share of output is saved — at t=20, k̃ is still at the old steady state so ỹ is unchanged, but (1−s) falls, so c̃ = (1−s)ỹ drops immediately). Over the transition, as k̃ rises, ỹ rises too, and c̃ eventually settles at the new steady state c̃\*_new.  
  **Golden Rule check:** c̃\* rises after an increase in s if and only if the economy is **below the Golden Rule** savings rate. The Golden Rule condition is s < α (capital share). Since the calibrated s ≈ 0.21–0.23 < α ≈ 0.30–0.35 for Denmark, the economy is indeed below the Golden Rule, confirming that c̃\* rises after the shock.

5. **Technology** is **unaffected** by the shock, confirming the exogenous nature of TFP in the Solow model.

### Key Takeaway
The General Solow Model predicts that Norway's fiscal rule tightening (higher oil fund reinvestment) (higher s) would raise the long-run **level** of output and consumption per worker, but not the **growth rate**. For sustained higher growth, Norway would need policies that increase the rate of technological progress g — especially important given the long-run depletion of oil reserves — for example, R&D subsidies, education investment, or openness to trade (Lecture 4, determinants of technological progress).